# 09 · Consistency & Consensus — Agreement Despite Everything

**Part 1 · Data-Intensive Applications** · *Prerequisites: 05, 08* · *Book mapping: DDIA ch. 9*

Notebook 08 was the demolition; this is the construction. Three artifacts get built today:
a **linearizability checker** (so "strongly consistent" stops being hand-waving and becomes
a property you can test), **logical clocks** (order without trusting physical time), and a
toy **Raft** — leader election plus replicated log — which we then partition, and watch
*stay safe*.

**Learning objectives.**

1. Define linearizability precisely, check histories for it mechanically, and separate it
   from serializability.
2. State CAP without the folklore.
3. Build Lamport clocks (total order consistent with causality) and vector clocks (actual
   concurrency detection) and know which problem each solves.
4. Explain why total order broadcast ≡ consensus, then build Raft-style election and log
   replication; verify election safety and log safety under partition, crash, and zombie
   leadership.

In [ ]:
import numpy as np

from sysdes import viz
from sysdes.sim import Network, Node, Sim

viz.use_style()

## 1. Linearizability — "behaves like one copy"

> A history of operations (each an *interval*: invocation → response) on an object is
> **linearizable** if every operation can be assigned a single *linearization point* inside
> its interval such that the resulting sequential execution is legal (reads return the
> latest preceding write) — and crucially, if operation A **completed before** B began,
> A's point comes before B's. It is a **recency guarantee**: once anyone has seen a value,
> everyone subsequently sees it (or newer).

Do not confuse it with **serializability** (nb 07): serializability orders *transactions*
in *some* order — even one that disagrees with real time; linearizability constrains
*single-object* operations to respect *real-time* order. (Strong systems offering both are
called *strict serializable*.)

Definitions with quantifiers deserve executable form. A checker: try to order the
operations, respecting real-time precedence, such that register semantics hold — depth-first
with memoized dead ends (this is the algorithmic core of real tools like Jepsen's checkers):

In [ ]:
def linearizable(history: list[tuple], initial=None) -> bool:
    """history: list of (start, end, kind, value); kind in {'write','read'}.
    True iff some legal sequential order exists that respects real time."""
    failures: set[tuple] = set()

    def dfs(remaining: frozenset, register) -> bool:
        if not remaining:
            return True
        state = (remaining, register)
        if state in failures:
            return False
        for i in remaining:
            start_i = history[i][0]
            # Real-time rule: i can be next only if no other pending op already
            # FINISHED before i even STARTED (that op must come first).
            if any(history[j][1] < start_i for j in remaining if j != i):
                continue
            _, _, kind, value = history[i]
            if kind == "write":
                if dfs(remaining - {i}, value):
                    return True
            elif value == register:                  # a read must see the register
                if dfs(remaining - {i}, register):
                    return True
        failures.add(state)
        return False

    return dfs(frozenset(range(len(history))), initial)

# Self-test on trivial histories:
assert linearizable([(0, 1, "write", "a"), (2, 3, "read", "a")])
assert not linearizable([(0, 1, "write", "a"), (2, 3, "read", "zzz")])
# A read OVERLAPPING a write may see either old or new - both linearizable:
assert linearizable([(0, 4, "write", "new"), (1, 2, "read", None)], initial=None)
assert linearizable([(0, 4, "write", "new"), (1, 2, "read", "new")])
print("checker self-tests pass")

Now feed it the **quorum anomaly from nb 05** — Alice read the new value, then Bob
(starting *after* Alice finished) read the old one, while the write was still spreading:

In [ ]:
quorum_history = [
    (0, 70, "write", "v2"),     # the write, partially applied for a long time
    (10, 20, "read", "v2"),     # Alice: sees the new value
    (30, 40, "read", "v1"),     # Bob: starts AFTER Alice ends... sees the old one
]
assert not linearizable(quorum_history, initial="v1"), \
    "nb 05's quorum behavior is formally NOT linearizable"

# If Bob had also seen v2, the same shape is fine (write's point before Alice):
assert linearizable([(0, 70, "write", "v2"), (10, 20, "read", "v2"),
                     (30, 40, "read", "v2")], initial="v1")
print("the nb 05 anomaly is now a theorem, not an anecdote")

Who needs linearizability? Locks and leader election (two holders = disaster, nb 08),
uniqueness constraints (usernames, seat 12A), cross-channel coordination (the thumbnail
service must see the image the message told it about). Who doesn't? Almost everything else —
which is lucky, because the recency guarantee is expensive: every linearizable read must
coordinate (even quorum reads need read-repair-before-return tricks), and...

**CAP, stated properly.** When a network **P**artition happens (not *if*), each side must
choose: keep serving independently (**A**vailability — and give up linearizability), or
refuse/wait so the system still acts like one copy (**C**onsistency — and give up serving on
the minority side). That's all CAP says: *during a partition, pick one*. The interesting
everyday question (PACELC) is: **E**lse — with no partition — do you pay **L**atency for
consistency (coordinate every operation) or not? Cross-region linearizable writes cost a
WAN round trip *every time*, partition or no partition — nb 01's 140 ms, forever.

## 2. Ordering without clocks

Notebook 08 killed timestamp ordering. What replaces it: **logical clocks** — counters that
track *causality* (the **happens-before** relation: a → b if same-process-earlier, or a is
the send and b the receive of one message, or transitivity).

### 2.1 Lamport clocks — a total order consistent with causality

One integer per node: increment on every local event; stamp outgoing messages; on receive,
jump to `max(local, message) + 1`. Guarantee: **if a → b then LC(a) < LC(b)** (ties broken
by node name give a total order). *Not* guaranteed: the converse — concurrent events also
get ordered, arbitrarily but consistently. That's a feature for building total order, and a
limitation for detecting conflicts.

In [ ]:
class LamportNode(Node):
    def __init__(self, name):
        super().__init__(name)
        self.clock = 0
        self.events: list[tuple[str, int]] = []     # (label, lamport time)

    def local_event(self, label):
        self.clock += 1
        self.events.append((label, self.clock))

    def send_event(self, dst, label):
        self.clock += 1
        self.events.append((label, self.clock))
        self.send(dst, {"type": "msg", "lc": self.clock, "label": label})

    def handle(self, src, msg):
        self.clock = max(self.clock, msg["lc"]) + 1   # the Lamport jump
        self.events.append((f"recv:{msg['label']}", self.clock))

sim = Sim()
net = Network(sim, latency=(3.0, 6.0), seed=1)
A, B, C = LamportNode("A"), LamportNode("B"), LamportNode("C")
for n in (A, B, C):
    net.add(n)

# A causal chain across all three nodes, plus an unrelated concurrent event on C:
sim.at(1.0, A.send_event, "B", "m1")            # A -> B
sim.at(2.0, C.local_event, "c-solo")            # concurrent with everything above
sim.at(12.0, B.send_event, "C", "m2")           # B -> C (after receiving m1)
sim.run()

lc = {label: t for node in (A, B, C) for label, t in node.events}
print("lamport times:", lc)
assert lc["m1"] < lc["recv:m1"] < lc["m2"] < lc["recv:m2"], \
    "causal chains always climb in Lamport time"
# The concurrent event got SOME number - comparable, but meaninglessly so:
assert lc["c-solo"] < lc["m2"], "concurrent events are ordered too (arbitrarily)"
print("total order consistent with causality - but concurrency is invisible")

### 2.2 Vector clocks — detecting concurrency

One counter **per node**, carried as a vector. Now the order is *partial*: a ≤ b iff every
component of a ≤ b. Incomparable vectors = **concurrent** events — exactly the test nb 05's
siblings needed to distinguish "newer version" from "conflicting version".

In [ ]:
def vc_leq(a: dict, b: dict) -> bool:
    return all(v <= b.get(k, 0) for k, v in a.items())

def vc_compare(a: dict, b: dict) -> str:
    if vc_leq(a, b) and a != b:
        return "a happens-before b"
    if vc_leq(b, a) and a != b:
        return "b happens-before a"
    return "concurrent" if a != b else "equal"

def vc_merge(a: dict, b: dict) -> dict:
    return {k: max(a.get(k, 0), b.get(k, 0)) for k in a | b}

def vc_tick(vc: dict, me: str) -> dict:
    return {**vc, me: vc.get(me, 0) + 1}

# Two clients update a shopping cart through different replicas, concurrently:
alice_write = vc_tick({}, "A")                       # {A:1}
bob_write = vc_tick({}, "B")                         # {B:1}
assert vc_compare(alice_write, bob_write) == "concurrent", \
    "neither saw the other: TRUE CONFLICT -> keep both as siblings (nb 05)"

# Carol reads both siblings, merges the cart, writes back:
carol_write = vc_tick(vc_merge(alice_write, bob_write), "C")   # {A:1,B:1,C:1}
assert vc_compare(alice_write, carol_write) == "a happens-before b"
assert vc_compare(bob_write, carol_write) == "a happens-before b"
print(f"carol's write {carol_write} supersedes both siblings - safe to discard them")

# And a later independent write is again concurrent with nothing... check one:
dave_write = vc_tick({}, "D")
assert vc_compare(carol_write, dave_write) == "concurrent"
print("vector clocks: causality answered exactly; storage cost: one counter per writer")

The division of labor: **Lamport clocks** when you need *a* total order everyone agrees on
(timestamps for total order broadcast); **vector clocks** when you need to *detect
conflicts* (leaderless replication, nb 05). Neither needs a physical clock to be right.

## 3. Consensus — the keystone

> **Total order broadcast (TOB)**: messages delivered to every node, exactly once, *in the
> same order*. Give me TOB and I'll give you: a replicated state machine (feed the same
> ordered commands to copies), linearizable storage (writes = messages; reads = wait for
> your marker), distributed locks, unique usernames. Give me **consensus** (nodes agree on
> one value, safely) and I'll build TOB (agree on each next message). **The two are
> equivalent** — and both are exactly what a failing-over single-leader system was missing:
> "who is the leader?" *is* a consensus question, and nb 08 proved you can't answer it with
> timeouts and self-belief alone.

**Raft** (the understandable consensus protocol; Paxos's equal in power) in one breath:
time is divided into **terms**; at most one leader per term. Nodes are followers until
heartbeats stop; then a follower increments the term and campaigns; nodes grant **one vote
per term**, only to candidates whose **log is at least as up-to-date** as theirs; majority
of votes → leader. The leader sequences all writes into its log, replicates, and marks
entries **committed** once a majority holds them (only entries from its own term count for
this). Committed = applied = irrevocable. Safety through everything comes from two
overlapping-majorities facts: two candidates can't both win a term, and a new leader's
majority must include someone who has every committed entry.

Our toy makes two declared simplifications (fine at this scale, noted where they bite):
leaders ship their **entire log** in heartbeats (real Raft ships increments with a
consistency check), and nothing persists to disk (we crash nodes only by partition).

In [ ]:
HEARTBEAT = 5.0
ELECTIONS: list[tuple[int, str]] = []       # (term, leader) - global safety record

class RaftNode(Node):
    def __init__(self, name, cluster_size, timeout_rng):
        super().__init__(name)
        self.cluster_size = cluster_size
        self.majority = cluster_size // 2 + 1
        self.state = "follower"
        self.term = 0
        self.voted_for: dict[int, str] = {}          # term -> candidate voted for
        self.log: list[tuple[int, str]] = []         # (term, command)
        self.commit_index = 0                        # count of committed entries
        self.applied: list[str] = []                 # the state machine
        self.votes: set[str] = set()
        self.acks: dict[str, int] = {}               # leader: peer -> log length acked
        self._timer = None
        self._timeout = lambda: float(timeout_rng.uniform(15.0, 30.0))

    # --------------------------------------------------------------- timers
    def start(self):
        self._reset_timer()

    def _reset_timer(self):
        if self._timer:
            self._timer.cancel()
        self._timer = self.after(self._timeout(), self._campaign)

    # ------------------------------------------------------------- election
    def _campaign(self):
        if self.state == "leader":
            return
        self.term += 1
        self.state = "candidate"
        self.voted_for[self.term] = self.name
        self.votes = {self.name}
        last_term = self.log[-1][0] if self.log else 0
        self.broadcast({"type": "request_vote", "term": self.term, "cand": self.name,
                        "last_idx": len(self.log), "last_term": last_term})
        self._reset_timer()                          # if votes split, campaign again

    def _up_to_date(self, msg) -> bool:
        """Raft's log guard: never elect a leader missing committed entries."""
        my_last_term = self.log[-1][0] if self.log else 0
        return (msg["last_term"], msg["last_idx"]) >= (my_last_term, len(self.log))

    def _become_leader(self):
        self.state = "leader"
        self.acks = {}
        ELECTIONS.append((self.term, self.name))
        self._heartbeat()

    # ---------------------------------------------------------- replication
    def _broadcast_log(self):
        self.broadcast({"type": "append", "term": self.term,
                        "log": list(self.log), "commit": self.commit_index})

    def _heartbeat(self):
        if self.state != "leader":
            return                                   # deposed: stop the loop
        self._broadcast_log()
        self.after(HEARTBEAT, self._heartbeat)

    def client_command(self, cmd: str):
        assert self.state == "leader", f"{self.name} is not the leader"
        self.log.append((self.term, cmd))
        self._broadcast_log()

    def _advance_commit(self):
        """Commit the longest prefix replicated on a majority - counting only
        entries from MY term (committing older-term entries by counting is the
        famous figure-8 unsafety in the Raft paper)."""
        for n in range(len(self.log), self.commit_index, -1):
            holders = 1 + sum(1 for m in self.acks.values() if m >= n)
            if holders >= self.majority and self.log[n - 1][0] == self.term:
                self.commit_index = n
                break
        self._apply()

    def _apply(self):
        while len(self.applied) < self.commit_index:
            self.applied.append(self.log[len(self.applied)][1])

    # -------------------------------------------------------------- messages
    def handle(self, src, msg):
        if msg.get("term", 0) > self.term:           # someone lives in the future:
            self.term = msg["term"]                  # adopt their term, step down
            self.state = "follower"
            self._reset_timer()

        kind = msg["type"]
        if kind == "request_vote":
            grant = (msg["term"] == self.term
                     and self.voted_for.get(self.term) in (None, msg["cand"])
                     and self._up_to_date(msg))
            if grant:
                self.voted_for[self.term] = msg["cand"]
                self._reset_timer()                  # give the candidate a chance
                self.send(src, {"type": "vote", "term": self.term})

        elif kind == "vote":
            if self.state == "candidate" and msg["term"] == self.term:
                self.votes.add(src)
                if len(self.votes) >= self.majority:
                    self._become_leader()

        elif kind == "append":
            if msg["term"] < self.term:
                return                               # stale leader: ignore
            self.state = "follower"                  # current leader exists
            self._reset_timer()
            self.log = list(msg["log"])              # toy: adopt leader's log wholesale
            self.commit_index = max(self.commit_index, min(msg["commit"], len(self.log)))
            self._apply()
            self.send(src, {"type": "append_ack", "term": self.term,
                            "match": len(self.log)})

        elif kind == "append_ack":
            if self.state == "leader" and msg["term"] == self.term:
                self.acks[src] = msg["match"]
                self._advance_commit()

def raft_cluster(n=5, seed=0):
    ELECTIONS.clear()
    sim = Sim()
    net = Network(sim, latency=(1.0, 2.0), seed=seed)
    nodes = [RaftNode(f"n{i}", n, np.random.default_rng(seed * 100 + i))
             for i in range(n)]
    for node in nodes:
        net.add(node)
        sim.at(0.0, node.start)
    return sim, net, nodes

def leader_among(nodes):
    leaders = [n for n in nodes if n.state == "leader"]
    assert len(leaders) == 1, f"expected one leader, found {[n.name for n in leaders]}"
    return leaders[0]

print("raft ready")

### 3.1 Elections: one leader, then a crash, then another

In [ ]:
sim, net, nodes = raft_cluster(seed=4)
sim.run(until=100.0)

first = leader_among(nodes)
print(f"elected: {first.name} in term {first.term}")
assert all(n.term == first.term for n in nodes), "everyone converged on the term"

# The leader dies (isolated). The rest re-elect in a HIGHER term.
crash = [n.name for n in nodes if n is not first]
net.partition({first.name}, set(crash))
sim.run(until=250.0)

survivors = [n for n in nodes if n is not first]
second = leader_among(survivors)
print(f"after crash: {second.name} leads in term {second.term}")
assert second.term > first.term, "new leadership = new term - fencing tokens reborn!"

# ELECTION SAFETY, checked over every election that ever happened:
terms = [t for t, _ in ELECTIONS]
assert len(terms) == len(set(terms)), f"two leaders shared a term! {ELECTIONS}"
print(f"election history: {ELECTIONS} - at most one leader per term")

Notice what the term is: a **fencing token** (nb 08), grown into a protocol. Every message
carries it; anyone meeting a higher term instantly abandons old beliefs.

### 3.2 The money scenario: a partitioned, zombified leader — and safety anyway

Now the full drill. A committed write; then a partition traps the leader in the minority;
clients keep writing to it (it *believes* it leads — nb 08's zombie, alive and well); the
majority elects a new leader and commits new writes; the partition heals.

In [ ]:
sim, net, nodes = raft_cluster(seed=11)
sim.run(until=100.0)
old_leader = leader_among(nodes)

# A write that COMMITS while the cluster is healthy:
old_leader.client_command("pay alice $5")
sim.run(until=140.0)
assert old_leader.commit_index == 1 and old_leader.applied == ["pay alice $5"]

# Partition: old leader + one loyal follower (minority) vs the other three.
loyal = next(n for n in nodes if n is not old_leader)
minority = {old_leader.name, loyal.name}
majority_nodes = [n for n in nodes if n.name not in minority]
net.partition(minority, {n.name for n in majority_nodes})

# The zombie side takes a client write... and can never commit it (2 < 3):
sim.at(150.0, old_leader.client_command, "pay bob $9 (doomed)")
# The majority side elects a new leader and takes a write that DOES commit:
sim.run(until=280.0)
new_leader = leader_among(majority_nodes)
new_leader.client_command("pay carol $7")
sim.run(until=340.0)

assert old_leader.state == "leader", "the zombie still believes"
assert old_leader.commit_index == 1, "believing is not committing: no majority, no commit"
assert new_leader.term > old_leader.term
assert new_leader.applied == ["pay alice $5", "pay carol $7"], \
    "the majority side carries all committed history forward"

# Heal. Higher term arrives; the zombie steps down and adopts the true log.
net.heal()
sim.run(until=420.0)

logs = {n.name: [cmd for _, cmd in n.log] for n in nodes}
print("final logs:", logs)
assert all(n.applied == ["pay alice $5", "pay carol $7"] for n in nodes), \
    "all five state machines identical"
assert all("doomed" not in " ".join(n.applied) for n in nodes), \
    "the uncommitted zombie write was discarded - never applied ANYWHERE"
terms = [t for t, _ in ELECTIONS]
assert len(terms) == len(set(terms)), "election safety held throughout"
print("SAFETY: committed writes survived; unacknowledged ones vanished cleanly; "
      "no term ever had two leaders")

Read the asserts as the contract consensus signs: **nothing acknowledged is ever lost;
nothing unacknowledged is ever half-visible; no two leaders share a term** — through
partitions, zombies, and re-elections. This is the machinery that finally closes nb 05's
failover horror story and nb 08's epistemic despair. Its price: every commit is a majority
round trip (latency), a minority partition cannot write (CAP's C branch, chosen), and the
protocol's subtleties are legion — which is why you deploy **ZooKeeper/etcd** (consensus as
a service: locks, leases, leader election, config — the coordination backbone nb 06's
routing map wanted) or a Raft-inside database, and do not write this yourself for
production.

**And 2PC?** Two-phase commit solves a different problem — *atomic commit across
partitions/systems* (all-or-nothing, nb 07's A distributed) — and does it with a
**blocking** flaw: participants that voted "yes" must hold locks and wait, potentially
forever, for a crashed coordinator's verdict. Consensus fixes exactly that (replicate the
verdict; a majority can always produce it), which is why modern distributed transactions
run 2PC *over* Paxos/Raft-replicated participants (Spanner's design).

## 4. Exercises

1. **Checker workout.** Decide by hand, then verify with `linearizable(...)`:
   (a) `W(a)[0,2] W(b)[1,3] R=a[4,5]`; (b) `W(a)[0,2] W(b)[3,5] R=a[6,7]`;
   (c) `W(a)[0,10] R=a[2,3] R=None[4,5]` with initial None.
2. **Read scaling.** Your service does 50k reads/s, 100 writes/s on config data and
   currently routes every read through the Raft leader for linearizability. Propose two
   designs that offload reads and state exactly what guarantee each read loses or keeps
   (leases/read-index vs follower reads).
3. **Figure 8, in our code.** Explain what could go wrong if `_advance_commit` dropped the
   `self.log[n-1][0] == self.term` condition — construct the danger story in terms of a
   deposed leader's entries being "majority-replicated" yet later overwritten.
4. **Split votes.** With identical (non-random) election timeouts, describe the livelock.
   Where exactly does our randomized `_timeout` break the symmetry, and what does raising
   the timeout range trade against detection latency (nb 08)?
5. **Build on TOB.** Using total order broadcast as a primitive, design linearizable
   compare-and-set: what do you broadcast, when do you decide the CAS won, and why must
   even *reads* consult the log's order (hint: zombie readers)?

### Solutions

**1.** (a) Not linearizable: R starts after both writes finished; W(b) overlaps W(a) but
completes at 3 < 4, so *some* write is last — trying orders: a,b → R=a fails; b,a → R=a
succeeds? b ends 3 > a's start 0 — real-time allows W(b) before W(a)? W(b) starts at 1 <
W(a) end 2 → they overlap → either order allowed → order b,a gives register=a → R=a ✓
linearizable! (The checker will say True — overlapping writes may linearize either way.)
(b) Not linearizable: W(b) finished before R began, so b must precede R; R=a fails in every
legal order. (c) Linearizable: R=a at point ~2.5 (inside W), then R=None... None must see
initial — but a read after a's point can't see None → try R=None first (point before W's
point, which W's [0,10] window allows), then R=a? Real time: R=a[2,3] ended before
R=None[4,5] began → R=a must come first → contradiction → **not** linearizable. Verified in
code below.

**2.** (i) **Leader leases + ReadIndex**: leader confirms leadership (majority heartbeat or
clock-bounded lease), then serves reads locally — keeps linearizability, costs leader
capacity and (for leases) a clock assumption (nb 08 flashbacks). (ii) **Follower reads**:
serve from any replica — infinitely scalable, but the guarantee drops to
"committed-but-possibly-stale" (bounded staleness if the follower gates on a commit index
the client carries — your nb 05 LSN trick, promoted to consensus).

**3.** A leader from term 2 replicates entry X to a majority but crashes before advancing
commit; term-3 leader (which lacks X — possible if X's majority didn't include it... it
must by up-to-date check — subtle: up-to-date compares last entry, and a longer-but-older
log can lose) writes Y at same index region; if a later term-2-style count had committed X
merely on replica count, Y's propagation would overwrite a "committed" X. Requiring a
current-term entry on top means the new leader first commits something of its own term,
which (by log adoption) pins every earlier entry beneath it — commitment becomes
irrevocable exactly once it's anchored by the current term.

**4.** All followers time out together, all campaign, all vote for themselves, nobody gets
a majority, repeat — forever, in lockstep. Randomization staggers the timeouts so one
candidate usually solicits votes before rivals wake. Wider ranges make split votes rarer
but leave the cluster leaderless longer after real failures — the nb 08 timeout dilemma
wearing a crown.

**5.** Broadcast `CAS(key, expect, new)` as a message; every replica applies messages in
TOB order, so the *first* CAS in the total order whose `expect` matches wins —
deterministically, on every replica. Reads: broadcast a read marker (or ReadIndex) and
answer only when your replica has applied up to it — otherwise a paused replica serves you
the past (a zombie *reader*: nb 08's lesson applies to reads too).

In [ ]:
# Exercise 1, checker-verified:
ex_a = [(0, 2, "write", "a"), (1, 3, "write", "b"), (4, 5, "read", "a")]
ex_b = [(0, 2, "write", "a"), (3, 5, "write", "b"), (6, 7, "read", "a")]
ex_c = [(0, 10, "write", "a"), (2, 3, "read", "a"), (4, 5, "read", None)]
assert linearizable(ex_a), "(a) overlapping writes may order either way"
assert not linearizable(ex_b), "(b) completed write cannot be unseen"
assert not linearizable(ex_c, initial=None), "(c) values cannot go backwards"
print("exercise 1 verified by the checker")

## 5. Takeaways

- **Linearizability** = single-copy behavior with real-time recency; it is a *testable*
  property (you built the test). Pay for it only where two-holders/uniqueness/recency truly
  matter; PACELC prices it even without partitions.
- **Lamport clocks**: total order consistent with causality (for sequencing). **Vector
  clocks**: partial order that exposes concurrency (for conflict detection). Physical time
  required: none.
- **Consensus ≡ total order broadcast** ≡ the missing piece of failover, locks, uniqueness,
  and atomic commit's blocking problem. Raft: terms (fencing tokens with a constitution),
  one vote per term, log-guarded elections, majority commit anchored in the current term.
- The contract: acknowledged writes survive anything short of losing a majority;
  unacknowledged writes vanish cleanly; leadership is never ambiguous within a term. Use
  ZooKeeper/etcd rather than reimplementing it.

**Further reading.** DDIA ch. 9; Ongaro & Ousterhout, *In Search of an Understandable
Consensus Algorithm* (Raft, 2014) — genuinely readable; Herlihy & Wing, *Linearizability*
(1990); Kingsbury's Jepsen analyses (jepsen.io) — this notebook's checker, industrialized.

**Next:** [10 · Batch processing](10-batch-processing.ipynb) — from guarding single values
to transforming entire datasets: MapReduce and the dataflow that grew out of it.